# Polaris RBAC Test

**Purpose:** Verify RBAC is correctly enforced across principals  
**Tests:** principal access boundaries, privilege enforcement

## Test sequence
```
R01: Read-only principal cannot write
R02: Write principal can read + write own catalog
R03: No cross-catalog access without explicit grant
R04: Deleted principal role → immediate 401
R05: Privilege minimum set correct (NAMESPACE_LIST)
```

In [ ]:
import sys
sys.path.insert(0, '.')
from polaris_test_utils import *

tr = TestResult()
token = get_token()
print("✅ Watchdog token acquired")

In [ ]:
# R01: Read-only cannot write
print("\nR01: Read-only principal cannot write to bronze")
# watchdog has read-only on watchdog catalog, not bronze
r = create_namespace(token, "bronze", "watchdog-test-ns")
passed = r.status_code in [403, 404]
tr.record("R01_readonly_cannot_write", passed,
    f"HTTP {r.status_code} ({'blocked' if passed else 'NOT BLOCKED — RBAC issue'})")

In [ ]:
# R02: Write principal can read + write watchdog catalog
print("\nR02: Write access to watchdog catalog")
# Create
r_create = create_namespace(token, WATCHDOG_CATALOG, "rbac-test-ns")
tr.record("R02a_can_write", r_create.status_code == 200, f"HTTP {r_create.status_code}")

# Read
r_read = get_namespace(token, WATCHDOG_CATALOG, "rbac-test-ns")
tr.record("R02b_can_read", r_read.status_code == 200, f"HTTP {r_read.status_code}")

tr.record("R02_write_access", r_create.status_code == 200 and r_read.status_code == 200,
    "read + write working")

In [ ]:
# R03: No cross-catalog access
print("\nR03: Cross-catalog access blocked")
# Try to list tables in gold (watchdog has no gold access)
r = requests.get(
    f"{POLARIS_URL}/api/catalog/v1/namespaces",
    headers={**polaris_headers(token), "warehouse": "gold"}
)
passed = r.status_code in [403, 404]
tr.record("R03_no_cross_catalog", passed,
    f"HTTP {r.status_code} ({'blocked' if passed else 'NOT BLOCKED'})")

In [ ]:
# R04: NAMESPACE_LIST privilege works
print("\nR04: Can list namespaces in watchdog catalog")
r = requests.get(
    f"{POLARIS_URL}/api/catalog/v1/namespaces",
    headers={**polaris_headers(token), "warehouse": WATCHDOG_CATALOG}
)
passed = r.status_code == 200
detail = f"HTTP {r.status_code}"
if passed:
    detail = f"namespaces visible: {r.json().get('namespaces', [])}"
tr.record("R04_namespace_list_works", passed, detail)

In [ ]:
# Cleanup
delete_namespace(token, WATCHDOG_CATALOG, "rbac-test-ns")

# Summary
print()
all_passed = tr.summary()
if not all_passed:
    print()
    print("⚠️  RBAC issues detected")
    print("   R01/R03 fail: principal has too many privileges")
    print("   R02 fail:     principal missing required privileges")
    print("   R04 fail:     NAMESPACE_LIST not granted at catalog level")